# 01 · vLLM debug-set run (Lab 7, Prompt 08)

Run by Claude Code via the Kaggle CLI (GPU T4 x2, Internet on). **Debug set only (25 items), never test items.**

1. Install vLLM 0.32.0 and print the resulting versions.
2. Clone the Lab 7 repository (configs, debug IDs).
3. Write and run `run_vllm_debug.py`: greedy (twice, for determinism) + 10 sampled runs per item, with the
   protocol's prompt, parser and decoding settings.

Outputs: `debug_summary.json` (no item text) and `debug_generations.jsonl` (quotes items; kept private).
These numbers check the pipeline; they are not study results.

In [ ]:
# Cell 1 - install vLLM (pinned) and show versions
import subprocess
r = subprocess.run("pip install -q vllm==0.32.0 2>&1 | tail -15", shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr)
# Fix from run 01 (debug log case 1): vllm upgrades torch to a CUDA 13.0 build but leaves Kaggle's
# torchaudio/torchao built for the old torch; transformers then imports torchaudio and fails. Neither
# package is needed for text generation, so remove them.
print(subprocess.run("pip uninstall -y -q torchaudio torchao 2>&1 | tail -3", shell=True, capture_output=True, text=True).stdout)
# Fail fast: import the vLLM entry points used later.
print(subprocess.run("python -c \"import vllm, torch, transformers, sys; from vllm import LLM, SamplingParams; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda, '| transformers', transformers.__version__, '| python', sys.version.split()[0], '| LLM import OK')\"",
                     shell=True, capture_output=True, text=True))

In [ ]:
# Cell 2 - get the code
print(subprocess.run("rm -rf /kaggle/working/repo && git clone --depth 1 https://github.com/vibhum-phd/lab7-oncology-llm-voting.git /kaggle/working/repo && "
                     "git -C /kaggle/working/repo log -1 --format='commit %H'", shell=True, capture_output=True, text=True).stdout)

In [ ]:
%%writefile /kaggle/working/run_vllm_debug.py
"""vLLM debug-set run (Prompt 08): pins the engine and checks prompt, parser and max_tokens on the
25 debug items only. Writes generations (git-ignored: they quote item text) and a text-free summary."""
import json, re, sys, time
import pandas as pd, yaml
from huggingface_hub import hf_hub_download


def main(rev: str) -> None:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    import vllm, torch, transformers
    root = "/kaggle/working/repo"
    cfg = yaml.safe_load(open(f"{root}/configs/protocol_v1.yaml"))
    paths = [hf_hub_download("GBaker/MedQA-USMLE-4-options-hf", f"default/{s}/0000.parquet",
                             repo_type="dataset", revision="refs/convert/parquet")
             for s in ("train", "validation", "test")]
    items = pd.concat([pd.read_parquet(p) for p in paths]).set_index("id")
    ids = open(f"{root}/data/splits/debug_ids.txt").read().split()
    name, d, p = cfg["model"]["name"], cfg["decoding"], cfg["prompt"]
    tok = AutoTokenizer.from_pretrained(name, revision=rev)

    def prompt(iid):
        r = items.loc[iid]
        user = p["user_template"].format(question=r.sent1, option_a=r.ending0, option_b=r.ending1,
                                         option_c=r.ending2, option_d=r.ending3)
        msgs = [{"role": "system", "content": p["system"]}, {"role": "user", "content": user}]
        return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)

    prompts = [prompt(i) for i in ids]
    prompt_lens = [len(tok(x)["input_ids"]) for x in prompts]
    P, F = re.compile(cfg["parsing"]["primary_regex"]), re.compile(cfg["parsing"]["fallback_regex"])
    parse = lambda t: ((P.findall(t) or F.findall(t)) or [None])[-1]

    t0 = time.time()
    llm = LLM(model=name, revision=rev, dtype="float16", tensor_parallel_size=2, max_model_len=2048,
              gpu_memory_utilization=0.90, enforce_eager=True, disable_custom_all_reduce=True,
              seed=20261102)
    load_s = time.time() - t0
    greedy = SamplingParams(temperature=0.0, max_tokens=d["max_tokens"],
                            repetition_penalty=d["repetition_penalty"])
    s = d["sampled"]
    sampled = [SamplingParams(n=s["n"], temperature=s["temperature"], top_p=s["top_p"], top_k=s["top_k"],
                              max_tokens=d["max_tokens"], repetition_penalty=d["repetition_penalty"],
                              seed=20261102 + k) for k in range(len(ids))]
    rows, timing = [], {}
    for arm, params in (("greedy", greedy), ("greedy_rerun", greedy), ("sampled", sampled)):
        t = time.time()
        outs = llm.generate(prompts, params)
        timing[arm] = time.time() - t
        for iid, o in zip(ids, outs):
            for j, c in enumerate(o.outputs):
                rows.append({"id": iid, "arm": arm, "run": j, "parsed": parse(c.text),
                             "key": "ABCD"[int(items.loc[iid].label)], "new_tokens": len(c.token_ids),
                             "finish": c.finish_reason, "text": c.text})
    with open("/kaggle/working/debug_generations.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

    df = pd.DataFrame(rows)
    g1 = df[df.arm == "greedy"].set_index("id")
    g2 = df[df.arm == "greedy_rerun"].set_index("id")
    sm = df[df.arm == "sampled"]
    vote = sm.dropna(subset=["parsed"]).groupby("id")["parsed"].agg(lambda x: x.value_counts().index[0])
    summary = {
        "versions": {"vllm": vllm.__version__, "torch": torch.__version__, "transformers": transformers.__version__,
                     "python": sys.version.split()[0]},
        "model_revision": rev, "n_items": len(ids), "load_seconds": round(load_s, 1),
        "prompt_tokens_max": max(prompt_lens), "prompt_tokens_mean": round(sum(prompt_lens) / len(prompt_lens), 1),
        "generations": {a: int((df.arm == a).sum()) for a in timing},
        "seconds": {a: round(v, 1) for a, v in timing.items()},
        "output_tokens": {a: int(df[df.arm == a].new_tokens.sum()) for a in timing},
        "tokens_per_second": {a: round(df[df.arm == a].new_tokens.sum() / timing[a], 1) for a in timing},
        "new_tokens_mean_max": {a: [round(df[df.arm == a].new_tokens.mean(), 1), int(df[df.arm == a].new_tokens.max())] for a in timing},
        "unparseable": {a: int(df[df.arm == a].parsed.isna().sum()) for a in timing},
        "truncated_length": {a: int((df[df.arm == a].finish == "length").sum()) for a in timing},
        "greedy_rerun_identical_text": int((g1.text == g2.loc[g1.index].text).sum()),
        "greedy_rerun_identical_letter": int((g1.parsed == g2.loc[g1.index].parsed).sum()),
        "pipeline_check_only__accuracy_greedy": int((g1.parsed == g1.key).sum()),
        "pipeline_check_only__accuracy_vote": int((vote == g1.loc[vote.index].key).sum()),
        "pipeline_check_only__items_unanimous_10of10": int(sm.groupby("id").parsed.agg(lambda x: x.nunique(dropna=False) == 1).sum()),
    }
    json.dump(summary, open("/kaggle/working/debug_summary.json", "w"), indent=1)
    print(json.dumps(summary, indent=1))


if __name__ == "__main__":
    main(sys.argv[1])

In [ ]:
# Cell 4 - run (model revision pinned from run 00)
r = subprocess.run("cd /kaggle/working && python run_vllm_debug.py a09a35458c702b33eeacc393d103063234e8bc28 2>&1 | tail -120",
                   shell=True, capture_output=True, text=True)
print(r.stdout[-20000:], r.stderr[-3000:])
print(subprocess.run("pip freeze > /kaggle/working/pip_freeze.txt; nvidia-smi --query-gpu=memory.used --format=csv", shell=True, capture_output=True, text=True).stdout)